# Proyecto 2 – Visión artificial
## Reconocimiento de texturas con *fine-tuning* de redes preentrenadas (dataset DTD)

**Universidad de Medellín · Línea de Énfasis III: Inteligencia Artificial**

**Grupo:** _[número]_  
**Integrantes:** _[nombre 1]_ · _[nombre 2]_

---

**Resumen.** Se entrena un clasificador de imágenes que reconoce **47 tipos de textura** (rayado, agrietado, tejido, escamoso, etc.) usando el dataset `dtd` del repositorio de TensorFlow Datasets. Se aplica *fine-tuning* sobre dos modelos fundacionales preentrenados en ImageNet (**ResNet50** y **VGG16**), se prueba una malla de hiperparámetros para cada uno y se selecciona el mejor modelo con el conjunto de validación. El conjunto de prueba se usa **una sola vez**, al final.

### Estructura del notebook (relación con el enunciado)

| Sección | Qué contiene | Punto del enunciado |
|---|---|---|
| 1 | Descripción del dataset: dominio, utilidad, variable de salida, tipo de problema | 3 |
| 2 | Configuración y carga del dataset desde TensorFlow Datasets | 1 y 2 |
| 3 | Análisis exploratorio con imágenes de todas las clases | 4 |
| 4 | Pipeline de datos y definición de los modelos (*fine-tuning*) | 5 |
| 5 | Búsqueda en malla de hiperparámetros con 2 arquitecturas | 6 |
| 6 | Mejor modelo: evaluación final, matriz de confusión y análisis de errores | 7 |
| 7 | Conclusiones | – |

> **Antes de ejecutar en Google Colab:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*. Con CPU el entrenamiento es demasiado lento.

## 1. Descripción del dataset

**Nombre en TensorFlow Datasets:** `dtd` (*Describable Textures Dataset*, Cimpoi et al., CVPR 2014).  
Catálogo: https://www.tensorflow.org/datasets/catalog/dtd

| Característica | Detalle |
|---|---|
| Contenido | Fotografías de texturas tomadas de la web (Google y Flickr), a color (RGB) |
| Tamaño | 5.640 imágenes, 120 por clase |
| Clases | 47 categorías de textura descritas con adjetivos (*banded, cracked, woven, scaly, …*) |
| Resolución | Variable: entre 300×300 y 640×640 píxeles |
| Particiones | `train`, `validation` y `test` oficiales, de 1.880 imágenes cada una (40 por clase). La celda de carga lo verifica. |

**Dominio del problema.** Visión por computador aplicada al reconocimiento de **texturas y materiales**: en vez de reconocer "qué objeto es", el modelo debe reconocer "cómo se ve la superficie" (a rayas, con grietas, tejida, con lunares…).

**Utilidad.**
- Control de calidad industrial: detectar patrones o defectos en superficies como telas, cerámica o metal.
- Robótica y manipulación: inferir el tipo de superficie o material antes de agarrar o caminar sobre ella.
- Moda y diseño: buscar o catalogar imágenes por patrón.
- Apoyo a otros sistemas de visión, porque la textura es una pista clave para distinguir materiales.

**Variable de salida.** Una etiqueta categórica con una de las **47 clases** de textura (variable `label`, entero de 0 a 46).

**Tipo de problema.** **Clasificación supervisada multiclase**: una imagen entra, una sola clase sale. Las clases están balanceadas, por eso la **exactitud (*accuracy*)** es una métrica válida como criterio principal. Se complementa con precisión, *recall* y F1 por clase y con la matriz de confusión.

**¿Por qué es buen candidato para *fine-tuning*?** Hay solo 40 imágenes de entrenamiento por clase. Entrenar una CNN desde cero con tan pocos datos sobreajusta casi seguro. Una red preentrenada en ImageNet ya trae filtros de bordes, patrones y texturas que se reutilizan y solo se ajustan al problema.

## 2. Configuración y carga del dataset

**Parámetros que se pueden cambiar en esta celda:**
- `MODO_RAPIDO = True` ejecuta una prueba corta (1 combinación por arquitectura, pocas épocas) para verificar que todo funciona. Se deja en `False` para el entrenamiento real.
- `GRID` define la malla de hiperparámetros del punto 6 del enunciado.
- `IMG_SIZE = 224` es el tamaño de entrada con el que se entrenaron VGG16 y ResNet50 en ImageNet.

In [5]:
import time, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_datasets as tfds
from PIL import Image
import cv2
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU disponible:", gpus if gpus else "NINGUNA -> en Colab: Entorno de ejecución > Cambiar tipo > GPU T4")

TensorFlow: 2.22.0-rc0
GPU disponible: NINGUNA -> en Colab: Entorno de ejecución > Cambiar tipo > GPU T4


In [6]:
# ---------------- Configuración general ----------------
IMG_SIZE = 224            # tamaño de entrada esperado por VGG16 / ResNet50
WEIGHTS = "imagenet"      # pesos preentrenados (fundacionales)
MODO_RAPIDO = False       # True = prueba corta para verificar que el notebook corre

if MODO_RAPIDO:
    GRID = {"batch_size": [32], "lr_ft": [1e-4], "dense_units": [128]}
    EPOCHS_HEAD, EPOCHS_FT = 2, 2
else:
    # Malla de hiperparámetros (se prueba cada combinación en cada arquitectura)
    GRID = {
        "batch_size":  [16, 32],      # tamaño de lote
        "lr_ft":       [1e-4, 1e-5],  # tasa de aprendizaje durante el fine-tuning
        "dense_units": [128, 256],    # neuronas de la capa densa del clasificador
    }
    EPOCHS_HEAD, EPOCHS_FT = 4, 6     # épocas fase 1 (cabeza) y fase 2 (fine-tuning)

PATIENCE = 2                          # EarlyStopping: épocas sin mejorar antes de parar
DROPOUT = 0.3
print("Combinaciones por arquitectura:", int(np.prod([len(v) for v in GRID.values()])))

Combinaciones por arquitectura: 8


In [7]:
# ---------------- Carga desde TensorFlow Datasets (no se sube al repositorio) ----------------
# as_supervised=True devuelve tuplas (imagen, etiqueta)
try:
    (ds_train_raw, ds_val_raw, ds_test_raw), info = tfds.load(
        "dtd", split=["train", "validation", "test"], as_supervised=True, with_info=True
    )
except ValueError:
    # Respaldo: si el dataset no define as_supervised, se arma la tupla a mano
    (ds_train_raw, ds_val_raw, ds_test_raw), info = tfds.load(
        "dtd", split=["train", "validation", "test"], with_info=True
    )
    to_tuple = lambda d: (d["image"], d["label"])
    ds_train_raw, ds_val_raw, ds_test_raw = [d.map(to_tuple) for d in (ds_train_raw, ds_val_raw, ds_test_raw)]

class_names = info.features["label"].names
NUM_CLASSES = info.features["label"].num_classes
print("Número de clases:", NUM_CLASSES)
print("Imágenes por partición:", {k: v.num_examples for k, v in info.splits.items()})
print(class_names)

ModuleNotFoundError: No module named 'importlib_resources'

## 3. Análisis exploratorio de datos (EDA)

Objetivos: (1) comprobar que las clases están balanceadas, (2) **ver imágenes de todas las clases**, (3) medir el tamaño y el color de las imágenes para decidir el preprocesamiento.

In [ ]:
# 3.1 Balance de clases por partición
def contar_clases(ds):
    etiquetas = np.array([int(y) for y in ds.map(lambda x, y: y).as_numpy_iterator()])
    return np.bincount(etiquetas, minlength=NUM_CLASSES)

conteos = pd.DataFrame({
    "train": contar_clases(ds_train_raw),
    "validation": contar_clases(ds_val_raw),
    "test": contar_clases(ds_test_raw),
}, index=class_names)

print(conteos.describe().loc[["min", "max", "mean"]])
conteos.plot(kind="bar", figsize=(16, 4), width=0.8)
plt.title("Imágenes por clase y partición")
plt.ylabel("Número de imágenes")
plt.tight_layout()
plt.show()

In [ ]:
# 3.2 Una imagen de cada una de las 47 clases (reducidas con Pillow para dibujarlas rápido)
ejemplos = {}
for img, lbl in ds_train_raw.as_numpy_iterator():
    l = int(lbl)
    if l not in ejemplos:
        ejemplos[l] = Image.fromarray(img).resize((160, 160))
    if len(ejemplos) == NUM_CLASSES:
        break

cols = 8
filas = int(np.ceil(NUM_CLASSES / cols))
fig, axes = plt.subplots(filas, cols, figsize=(2.4 * cols, 2.6 * filas))
for ax in axes.ravel():
    ax.axis("off")
for l, im in sorted(ejemplos.items()):
    ax = axes.ravel()[l]
    ax.imshow(im)
    ax.set_title(class_names[l], fontsize=10)
plt.suptitle("Una imagen por clase (partición train)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 3.3 Variación dentro de una misma clase: 5 imágenes de 4 clases
nombres_demo = ["banded", "cracked", "woven", "scaly"]
clases_demo = [class_names.index(n) for n in nombres_demo if n in class_names] or [0, 1, 2, 3]
por_clase = {c: [] for c in clases_demo}
for img, lbl in ds_train_raw.as_numpy_iterator():
    l = int(lbl)
    if l in por_clase and len(por_clase[l]) < 5:
        por_clase[l].append(Image.fromarray(img).resize((200, 200)))
    if all(len(v) == 5 for v in por_clase.values()):
        break

fig, axes = plt.subplots(len(clases_demo), 5, figsize=(14, 3 * len(clases_demo)))
for i, c in enumerate(clases_demo):
    for j, im in enumerate(por_clase[c]):
        axes[i, j].imshow(im)
        axes[i, j].axis("off")
        if j == 0:
            axes[i, j].set_title(class_names[c], loc="left", fontsize=12)
plt.suptitle("Variabilidad dentro de la misma clase", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 3.4 Tamaño y color de las imágenes (NumPy)
alturas, anchos, medias_rgb = [], [], []
for img, _ in ds_train_raw.as_numpy_iterator():
    h, w, _ = img.shape
    alturas.append(h)
    anchos.append(w)
    medias_rgb.append(img.reshape(-1, 3).mean(axis=0))
medias_rgb = np.array(medias_rgb)

print(f"Alto  : min={min(alturas)}, max={max(alturas)}")
print(f"Ancho : min={min(anchos)}, max={max(anchos)}")
print("Media de cada canal (R, G, B):", medias_rgb.mean(axis=0).round(1))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].hist(alturas, bins=30, color="steelblue"); axes[0].set_title("Alto (px)")
axes[1].hist(anchos, bins=30, color="seagreen"); axes[1].set_title("Ancho (px)")
for k, (c, col) in enumerate(zip("RGB", ["red", "green", "blue"])):
    axes[2].hist(medias_rgb[:, k], bins=30, alpha=0.5, color=col, label=c)
axes[2].set_title("Media de intensidad por canal"); axes[2].legend()
plt.tight_layout()
plt.show()

In [ ]:
# 3.5 Qué le hace el preprocesamiento a una imagen (Pillow + OpenCV)
img_demo, lbl_demo = next(iter(ds_train_raw.as_numpy_iterator()))
orig = Image.fromarray(img_demo)
redim = orig.resize((IMG_SIZE, IMG_SIZE))                       # lo que verá la red
gris = cv2.cvtColor(np.array(redim), cv2.COLOR_RGB2GRAY)
bordes = cv2.Canny(gris, 100, 200)                              # los bordes son la base de la textura

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(orig); axes[0].set_title(f"Original {orig.size} - {class_names[int(lbl_demo)]}")
axes[1].imshow(redim); axes[1].set_title(f"Redimensionada {IMG_SIZE}x{IMG_SIZE}")
axes[2].imshow(bordes, cmap="gray"); axes[2].set_title("Bordes (Canny)")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

**Observaciones del EDA (completar con lo que muestren las gráficas al ejecutar):**
- Las clases están balanceadas: todas tienen el mismo número de imágenes por partición, así que no se necesitan pesos de clase ni remuestreo.
- Las imágenes tienen tamaños distintos, por eso hay que **redimensionarlas a 224×224** para formar lotes del mismo tamaño.
- Hay clases visualmente muy cercanas entre sí (por ejemplo, texturas de rejilla, malla y tejido), lo que anticipa confusiones entre ellas.

## 4. Pipeline de datos y modelos (*fine-tuning*)

### 4.1 Pipeline con `tf.data`
1. **Redimensionar** cada imagen a 224×224 y guardarla como `uint8` en **caché** (se hace una sola vez; así las 16 corridas de la malla no vuelven a decodificar los JPEG).
2. **Barajar** solo el entrenamiento, en cada época.
3. Agrupar en **lotes** del tamaño que indique la malla.
4. Aplicar el **`preprocess_input` propio de cada arquitectura**: ResNet50 y VGG16 esperan los píxeles normalizados de forma distinta, y usar el equivocado empeora el resultado.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def _resize(img, lbl):
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(tf.round(img), tf.uint8), lbl

# Imágenes ya redimensionadas y cacheadas en memoria (se comparten entre todas las corridas)
base_train = ds_train_raw.map(_resize, num_parallel_calls=AUTOTUNE).cache()
base_val   = ds_val_raw.map(_resize, num_parallel_calls=AUTOTUNE).cache()
base_test  = ds_test_raw.map(_resize, num_parallel_calls=AUTOTUNE).cache()

def make_datasets(preprocess_fn, batch_size):
    prep = lambda x, y: (preprocess_fn(tf.cast(x, tf.float32)), y)
    train = (base_train.shuffle(2000, seed=SEED, reshuffle_each_iteration=True)
             .batch(batch_size).map(prep, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE))
    val   = base_val.batch(batch_size).map(prep, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
    test  = base_test.batch(batch_size).map(prep, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
    return train, val, test

# "Calentar" la caché: recorre cada partición una vez (tarda ~1 minuto)
t0 = time.time()
for d in (base_train, base_val, base_test):
    for _ in d:
        pass
print(f"Caché lista en {time.time() - t0:.0f} s")

### 4.2 Arquitecturas y estrategia de *fine-tuning*

Se comparan **dos arquitecturas fundacionales** preentrenadas en ImageNet:

| | VGG16 | ResNet50 |
|---|---|---|
| Idea | Pila profunda de convoluciones 3×3 simples | Bloques con **conexiones residuales** que facilitan entrenar redes profundas |
| Parámetros del extractor | ~14,7 M | ~23,6 M |
| Último bloque convolucional | `block5_*` | `conv5_*` |

**Cabeza de clasificación** (igual para ambas): `GlobalAveragePooling2D → Dense(ReLU) → Dropout → Dense(47, softmax)`.
- *Global average pooling* reduce cada mapa de características a un número, lo que baja mucho los parámetros frente a `Flatten`.
- *Dropout* regulariza para reducir el sobreajuste con tan pocas imágenes.
- Softmax de 47 salidas porque es clasificación multiclase.

**Aumento de datos** (solo en entrenamiento): volteo horizontal y rotación/zoom pequeños. Crea variantes de las imágenes sin tener que etiquetar nada nuevo.

**Fine-tuning en dos fases:**
1. **Fase 1 – entrenar la cabeza:** el extractor está **congelado** y solo se entrenan las capas nuevas (`lr = 1e-3`). Así la cabeza, que parte de pesos aleatorios, no daña los pesos preentrenados con gradientes grandes.
2. **Fase 2 – ajuste fino:** se **descongela el último bloque convolucional** y se sigue entrenando con una tasa de aprendizaje baja (`lr_ft`, parte de la malla). Se ajustan las características más especializadas al dominio de texturas. Las capas de *BatchNorm* de ResNet50 se mantienen congeladas, que es la práctica recomendada para *fine-tuning*.

In [ ]:
ARQUITECTURAS = {
    "ResNet50": dict(builder=keras.applications.ResNet50,
                     prep=keras.applications.resnet50.preprocess_input,
                     ultimo_bloque="conv5_"),
    "VGG16":    dict(builder=keras.applications.VGG16,
                     prep=keras.applications.vgg16.preprocess_input,
                     ultimo_bloque="block5_"),
}

augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="augmentation")

def build_model(arch, dense_units, dropout=DROPOUT):
    cfg = ARQUITECTURAS[arch]
    base = cfg["builder"](include_top=False, weights=WEIGHTS, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False                                   # Fase 1: extractor congelado
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = augmentation(inputs)                                 # solo se activa al entrenar
    x = base(x, training=False)                              # BatchNorm en modo inferencia
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name=f"{arch}_finetune"), base

def descongelar_ultimo_bloque(base, prefijo):
    base.trainable = True
    for layer in base.layers:
        es_bn = isinstance(layer, layers.BatchNormalization)
        layer.trainable = layer.name.startswith(prefijo) and not es_bn

def contar_parametros(model):
    entrenables = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    return entrenables, int(model.count_params())

# Verificación: cuántos parámetros se entrenan en cada fase
for arch in ARQUITECTURAS:
    m, b = build_model(arch, 128)
    tr1, tot = contar_parametros(m)
    descongelar_ultimo_bloque(b, ARQUITECTURAS[arch]["ultimo_bloque"])
    tr2, _ = contar_parametros(m)
    print(f"{arch:9s} total={tot:>11,}  entrenables fase 1={tr1:>9,}  entrenables fase 2={tr2:>11,}")
    del m, b
keras.backend.clear_session()

## 5. Búsqueda en malla de hiperparámetros (2 arquitecturas)

Para cada arquitectura se prueban **todas las combinaciones** de la malla `GRID` (búsqueda en malla o *grid search*): tamaño de lote, tasa de aprendizaje del *fine-tuning* y número de neuronas de la capa densa.

- El criterio para elegir el mejor modelo es la **exactitud en validación**, no en prueba. Así el conjunto de prueba sigue siendo una medida honesta del desempeño final.
- `EarlyStopping` detiene cada fase si la exactitud de validación no mejora y recupera los mejores pesos.
- Solo se guardan en disco los pesos del mejor modelo encontrado hasta el momento.

In [ ]:
BEST_PATH = "mejor_modelo.weights.h5"
resultados = []
mejor = {"val_acc": -1.0}

def correr_experimento(arch, batch_size, lr_ft, dense_units):
    global mejor
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    cfg = ARQUITECTURAS[arch]
    train_ds, val_ds, _ = make_datasets(cfg["prep"], batch_size)
    model, base = build_model(arch, dense_units)
    loss = "sparse_categorical_crossentropy"
    stop = lambda: keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=PATIENCE,
                                                 restore_best_weights=True)
    t0 = time.time()

    # Fase 1: solo la cabeza
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=loss, metrics=["accuracy"])
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, callbacks=[stop()], verbose=0)
    val_acc_cabeza = model.evaluate(val_ds, verbose=0)[1]

    # Fase 2: descongelar el último bloque y ajustar con lr bajo
    descongelar_ultimo_bloque(base, cfg["ultimo_bloque"])
    model.compile(optimizer=keras.optimizers.Adam(lr_ft), loss=loss, metrics=["accuracy"])
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, callbacks=[stop()], verbose=0)
    val_loss, val_acc = model.evaluate(val_ds, verbose=0)

    hist = {k: h1.history[k] + h2.history[k] for k in h1.history}
    fila = dict(arquitectura=arch, batch_size=batch_size, lr_ft=lr_ft, dense_units=dense_units,
                val_acc_fase1=round(val_acc_cabeza, 4), val_acc=round(val_acc, 4),
                val_loss=round(val_loss, 4), epocas=len(hist["loss"]),
                minutos=round((time.time() - t0) / 60, 1))
    resultados.append(fila)

    if val_acc > mejor["val_acc"]:
        model.save_weights(BEST_PATH)
        mejor = dict(fila, val_acc=val_acc, hist=hist, fase2_inicio=len(h1.history["loss"]))
    print(fila)
    return fila

for arch in ARQUITECTURAS:
    print(f"\n=== {arch} ===")
    for bs, lr, du in itertools.product(GRID["batch_size"], GRID["lr_ft"], GRID["dense_units"]):
        correr_experimento(arch, bs, lr, du)

In [ ]:
# Tabla completa de resultados, ordenada por exactitud en validación
df = pd.DataFrame(resultados).sort_values("val_acc", ascending=False).reset_index(drop=True)
display(df)

print("\nMejor combinación por arquitectura:")
display(df.loc[df.groupby("arquitectura")["val_acc"].idxmax()].reset_index(drop=True))

In [ ]:
# Efecto del fine-tuning: exactitud tras la fase 1 (solo cabeza) vs. tras la fase 2 (ajuste fino)
resumen = df.groupby("arquitectura")[["val_acc_fase1", "val_acc"]].max()
resumen.columns = ["Solo cabeza (fase 1)", "Con fine-tuning (fase 2)"]
resumen.plot(kind="bar", figsize=(6, 4), ylim=(0, 1))
plt.title("Mejor exactitud en validación por arquitectura")
plt.ylabel("Exactitud")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Mejor modelo

Se reconstruye la mejor configuración, se cargan sus pesos guardados y se evalúa **una sola vez** en el conjunto de prueba.

In [ ]:
print("Mejor configuración encontrada:")
print({k: mejor[k] for k in ["arquitectura", "batch_size", "lr_ft", "dense_units", "val_acc"]})

arch_best = mejor["arquitectura"]
best_model, _ = build_model(arch_best, mejor["dense_units"])
best_model.load_weights(BEST_PATH)
best_model.compile(loss="sparse_categorical_crossentropy", metrics=["accuracy"])

_, _, test_ds = make_datasets(ARQUITECTURAS[arch_best]["prep"], mejor["batch_size"])
test_loss, test_acc = best_model.evaluate(test_ds, verbose=0)
print(f"\nExactitud en PRUEBA: {test_acc:.4f}   |   pérdida: {test_loss:.4f}")
print(f"Azar (47 clases): {1 / NUM_CLASSES:.4f}")

In [ ]:
# Curvas de entrenamiento del mejor modelo (la línea punteada marca el inicio del fine-tuning)
h = mejor["hist"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (k, titulo) in zip(axes, [("accuracy", "Exactitud"), ("loss", "Pérdida")]):
    ax.plot(h[k], label="entrenamiento")
    ax.plot(h["val_" + k], label="validación")
    ax.axvline(mejor["fase2_inicio"] - 0.5, color="gray", linestyle="--", label="inicio fine-tuning")
    ax.set_title(titulo); ax.set_xlabel("Época"); ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Predicciones sobre el conjunto de prueba
y_true = np.concatenate([y.numpy() for _, y in test_ds])
probs = best_model.predict(test_ds, verbose=0)
y_pred = probs.argmax(axis=1)

print(classification_report(y_true, y_pred, target_names=class_names, digits=3, zero_division=0))

In [ ]:
# Matriz de confusión y pares de clases más confundidos
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 10))
plt.imshow(cm, cmap="Blues")
plt.colorbar(label="Número de imágenes")
plt.xticks(range(NUM_CLASSES), class_names, rotation=90, fontsize=7)
plt.yticks(range(NUM_CLASSES), class_names, fontsize=7)
plt.xlabel("Clase predicha"); plt.ylabel("Clase real")
plt.title("Matriz de confusión (prueba)")
plt.tight_layout()
plt.show()

fuera_diag = cm.copy()
np.fill_diagonal(fuera_diag, 0)
top = np.dstack(np.unravel_index(np.argsort(-fuera_diag.ravel())[:8], fuera_diag.shape))[0]
print("Confusiones más frecuentes (real -> predicha):")
for r, p in top:
    if fuera_diag[r, p] > 0:
        print(f"  {class_names[r]:14s} -> {class_names[p]:14s}  {fuera_diag[r, p]} veces")

In [ ]:
# Ejemplos de imágenes mal clasificadas
imgs_test = np.concatenate([x for x, _ in base_test.batch(256).as_numpy_iterator()])
errores = np.where(y_pred != y_true)[0]
print(f"Imágenes mal clasificadas: {len(errores)} de {len(y_true)}")

muestra = errores[:8]
fig, axes = plt.subplots(2, 4, figsize=(14, 7.5))
for ax, i in zip(axes.ravel(), muestra):
    ax.imshow(imgs_test[i])
    ax.set_title(f"Real: {class_names[y_true[i]]}\nPred: {class_names[y_pred[i]]}", fontsize=10)
    ax.axis("off")
plt.suptitle("Errores del mejor modelo", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Predicción de una imagen de prueba con las 3 clases más probables
i = int(np.random.default_rng(SEED).integers(len(y_true)))
top3 = np.argsort(-probs[i])[:3]
plt.imshow(imgs_test[i]); plt.axis("off")
plt.title(f"Real: {class_names[y_true[i]]}")
plt.show()
for c in top3:
    print(f"  {class_names[c]:14s} {probs[i][c]:.1%}")

## 7. Conclusiones

In [ ]:
mejor_por_arq = df.loc[df.groupby("arquitectura")["val_acc"].idxmax()].set_index("arquitectura")
print("RESUMEN AUTOMÁTICO")
print("-" * 60)
print(f"Mejor modelo: {arch_best} (batch={mejor['batch_size']}, lr fine-tuning={mejor['lr_ft']}, "
      f"neuronas densas={mejor['dense_units']})")
print(f"Exactitud validación: {mejor['val_acc']:.1%}  |  Exactitud prueba: {test_acc:.1%}  "
      f"(azar: {1 / NUM_CLASSES:.1%})")
for arch, fila in mejor_por_arq.iterrows():
    ganancia = fila["val_acc"] - fila["val_acc_fase1"]
    print(f"{arch}: mejor val_acc={fila['val_acc']:.1%}; el fine-tuning aportó {ganancia:+.1%} "
          f"sobre entrenar solo la cabeza.")
print(f"Tiempo total de la malla: {df['minutos'].sum():.0f} min")

**Conclusiones (redactar con los números del resumen anterior):**

1. **Modelo seleccionado:** _[arquitectura y configuración ganadora, con su exactitud en prueba]_.
2. **Fine-tuning vs. solo cabeza:** _[cuánto mejoró la fase 2 respecto a la fase 1 y qué indica sobre la similitud entre ImageNet y las texturas]_.
3. **Comparación ResNet50 vs. VGG16:** _[cuál rindió mejor y a qué costo en tiempo y parámetros; relación con las conexiones residuales]_.
4. **Hiperparámetros:** _[qué tasa de aprendizaje, lote y neuronas funcionaron mejor y por qué]_.
5. **Errores:** _[qué clases se confunden más y por qué visualmente son parecidas]_.
6. **Limitaciones y trabajo futuro:** solo 40 imágenes de entrenamiento por clase; se podría probar descongelar más bloques, usar `EfficientNet` o aumentar el tamaño de la malla.

**Referencia.** Cimpoi, M., Maji, S., Kokkinos, I., Mohamed, S., Vedaldi, A. (2014). *Describing Textures in the Wild.* CVPR.